# Cellpose-SAM Batch Segmentation (Lightning AI version)

Runs Cellpose-SAM on ~6000 cell images stored in Google Drive and saves masks back to Drive.

**How this differs from the Colab version:**
- Uses `rclone` (not `google.colab.drive`) to access Google Drive
- Per-folder workflow: sync images down to Studio disk → run Cellpose → sync masks back up to Drive
- Path layout is Lightning-local (`/teamspace/studios/this_studio/...`), not `/content/...`

**One-time setup (run in Lightning terminal, NOT in this notebook):**
```bash
# 1. Install rclone
curl https://rclone.org/install.sh | sudo bash

# 2. Configure Google Drive remote — name it exactly `gdrive`
rclone config
#   n -> new remote
#   name: gdrive
#   storage: drive
#   client_id / client_secret: blank
#   scope: 1 (full access)
#   service_account_file: blank
#   advanced: n
#   auto config: n   <-- IMPORTANT, no browser on Lightning
#   open the URL it prints on your laptop, paste the auth code back
#   team drive: n
#   y, then q

# 3. Verify
rclone lsd gdrive:
```

**Resumable:** if the Studio disconnects, just re-run from cell 5 onward — already-processed masks are skipped, and only un-uploaded masks get pushed.

## 0. Pull latest code from GitHub

In [ ]:
import os
os.chdir(os.path.expanduser('~/Huang-Lab-Work'))
# SSH key on this Studio is already added to GitHub, so no token prompt needed
!git pull origin main
print('Done.')

## 1. Check runtime

In [ ]:
import torch

USE_GPU = torch.cuda.is_available()
if USE_GPU:
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('Expected: a few sec/image — all 6000 in well under an hour.')
else:
    print('No GPU detected — running on CPU.')
    print('Expected: ~30-120 sec/image. For 6000 images, plan multiple sessions.')
    print('Tip: open a GPU Studio in Lightning (Settings -> Compute) for much faster runs.')
    print('Skip-if-exists is enabled, so each new session resumes where the last left off.')

## 2. Verify rclone is set up

If this cell errors, finish the one-time setup at the top of the notebook before continuing.

In [ ]:
import subprocess

result = subprocess.run(['rclone', 'lsd', 'gdrive:'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'rclone not configured. Error:\n{result.stderr}')
print('rclone OK. Top-level Drive folders:')
print(result.stdout)

## 3. Install Python dependencies

In [ ]:
# Step 1: downgrade numpy to a version compatible with cellpose/scipy
!pip install -q "numpy<2.0" "cellpose" tifffile tqdm

# Step 2: restart the kernel so the new numpy is loaded
# After the restart, re-run from cell 4 (Config) onward — skip this cell.
import importlib.metadata
print(f"numpy version: {importlib.metadata.version('numpy')}")
print(f"cellpose version: {importlib.metadata.version('cellpose')}")
print('')
print('*** Now restart the kernel (Kernel -> Restart), then re-run from cell 4 onward. ***')

## 4. Config

`DRIVE_ROOT` is the path **inside your Google Drive** (used by rclone, relative to the `gdrive:` remote).
`LOCAL_ROOT` is the path **on this Studio's disk** where images get cached during processing.

In [ ]:
from pathlib import Path

# Path inside your Google Drive (relative to gdrive: remote root, i.e. 'My Drive')
DRIVE_ROOT = 'Fusion AI/Prof Huang Project/Cellpose feature extractions'

# Path on Lightning Studio's local disk (fast I/O, persistent across restarts)
LOCAL_ROOT = Path('/teamspace/studios/this_studio/cellpose_work')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Stiffness conditions — one per folder
CONDITIONS = [
    '260513_TC_Level',
    '260514_900kPa',
    '260516_5kPa',
    '260516_500kPa',
    '260521_150kPa',
    '260522_500kPa',
]

MODEL_TYPE    = 'cpsam'  # Cellpose-SAM
DIAMETER      = None     # None = auto-estimate per image
CHANNELS      = [0, 0]   # [0,0] = use all channels for SAM
SKIP_EXISTING = True     # set False to reprocess everything from scratch
BATCH_SIZE    = 8        # images processed per model.eval() call
                         # increase if you have more RAM/GPU; decrease if you get OOM errors
IMG_EXTS      = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}

print(f'Local working dir: {LOCAL_ROOT}')
print(f'Drive root        : gdrive:{DRIVE_ROOT}')
print(f'Conditions        : {len(CONDITIONS)}')

## 5. rclone sync helpers (Drive <-> Studio disk)

`pull_images_from_drive` mirrors a Drive folder of input images to local disk (only downloads missing/new files).
`push_masks_to_drive` mirrors a local folder of mask outputs back up to Drive.

In [ ]:
import subprocess

def _run_rclone(args: list) -> None:
    """Run an rclone command, streaming output, raising on failure."""
    print(f'  $ rclone {" ".join(args)}')
    result = subprocess.run(['rclone', *args, '--progress', '--transfers=8', '--checkers=16'],
                            capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f'rclone failed (exit {result.returncode})')

def pull_images_from_drive(condition: str) -> Path:
    """Download `imgs/<condition>/tiles/` from Drive to local disk. Returns local path."""
    remote = f'gdrive:{DRIVE_ROOT}/imgs/{condition}/tiles'
    local  = LOCAL_ROOT / 'imgs' / condition / 'tiles'
    local.mkdir(parents=True, exist_ok=True)
    _run_rclone(['copy', remote, str(local)])
    return local

def push_masks_to_drive(condition: str) -> None:
    """Upload local masks for a condition back to Drive (only new/changed files)."""
    local  = LOCAL_ROOT / 'masks' / condition
    remote = f'gdrive:{DRIVE_ROOT}/masks/{condition}'
    _run_rclone(['copy', str(local), remote])

print('rclone helpers defined.')

## 6. Load model (once — reused across all 6000 images)

In [ ]:
from cellpose import models

model = models.CellposeModel(gpu=USE_GPU, model_type=MODEL_TYPE)
print(f'Model loaded: {MODEL_TYPE}  |  GPU={model.gpu}')

## 7. Segmentation functions

In [ ]:
import numpy as np
import tifffile
from tqdm.notebook import tqdm


def _load_image(img_path: Path) -> np.ndarray:
    """Load image and ensure shape is (H, W, 3)."""
    img = tifffile.imread(str(img_path))
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.ndim == 3 and img.shape[0] in (1, 3):  # (C,H,W) -> (H,W,C)
        img = np.moveaxis(img, 0, -1)
        if img.shape[-1] == 1:
            img = np.concatenate([img, img, img], axis=-1)
    return img


def segment_folder(in_dir: Path, out_dir: Path, model) -> dict:
    """Process all images in in_dir in batches, writing masks to out_dir."""
    out_dir.mkdir(parents=True, exist_ok=True)

    all_paths = sorted([p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    if not all_paths:
        print(f'  [WARN] No images found in {in_dir}')
        return {'processed': 0, 'skipped': 0, 'failed': 0}

    todo, skipped_paths = [], []
    for p in all_paths:
        out_path = out_dir / f'{p.stem}_masks.tif'
        if SKIP_EXISTING and out_path.exists():
            skipped_paths.append(p)
        else:
            todo.append(p)

    processed = failed = 0
    skipped = len(skipped_paths)
    failed_files = []

    chunks = [todo[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    pbar = tqdm(total=len(todo), desc=in_dir.parent.name, unit='img')

    for chunk in chunks:
        imgs, paths = [], []
        for p in chunk:
            try:
                imgs.append(_load_image(p))
                paths.append(p)
            except Exception as e:
                print(f'  [FAIL load] {p.name}: {e}')
                failed_files.append(p.name)
                failed += 1
                pbar.update(1)

        if not imgs:
            continue

        try:
            masks_list, _, _ = model.eval(
                imgs,
                diameter=DIAMETER,
                channels=CHANNELS,
                normalize=True,
            )
        except Exception as e:
            print(f'  [FAIL batch] {[p.name for p in paths]}: {e}')
            failed += len(paths)
            failed_files += [p.name for p in paths]
            pbar.update(len(paths))
            continue

        for p, mask in zip(paths, masks_list):
            out_path = out_dir / f'{p.stem}_masks.tif'
            try:
                tifffile.imwrite(str(out_path), mask.astype(np.uint16), compression='lzw')
                processed += 1
            except Exception as e:
                print(f'  [FAIL save] {p.name}: {e}')
                failed += 1
                failed_files.append(p.name)
            pbar.update(1)

    pbar.close()
    if failed_files:
        print(f'  Failed files: {failed_files}')
    return {'processed': processed, 'skipped': skipped, 'failed': failed}


print('Functions defined.')

## 8. Run segmentation on all folders

For each condition:
1. **Pull** input images from Drive to Studio disk (only downloads missing files)
2. **Run** Cellpose on the local copies
3. **Push** the resulting masks back up to Drive (only uploads new ones)

If the Studio disconnects, just re-run this cell — `skip-if-exists` and `rclone copy` are both idempotent.

In [ ]:
import time

grand_total = {'processed': 0, 'skipped': 0, 'failed': 0}
t_start = time.time()

for condition in CONDITIONS:
    print(f'\n========== {condition} ==========')

    # 1. Pull images from Drive
    print(f'[1/3] Pulling images from Drive...')
    in_dir = pull_images_from_drive(condition)

    # 2. Segment
    print(f'[2/3] Running Cellpose...')
    out_dir = LOCAL_ROOT / 'masks' / condition
    t0 = time.time()
    stats = segment_folder(in_dir, out_dir, model)
    elapsed = time.time() - t0
    print(f'  processed={stats["processed"]}  skipped={stats["skipped"]}  '
          f'failed={stats["failed"]}  ({elapsed/60:.1f} min)')

    # 3. Push masks back to Drive
    print(f'[3/3] Pushing masks to Drive...')
    push_masks_to_drive(condition)

    for k in grand_total:
        grand_total[k] += stats[k]

total_elapsed = time.time() - t_start
print(f'\n========== DONE ==========')
print(f'Total processed : {grand_total["processed"]}')
print(f'Total skipped   : {grand_total["skipped"]}')
print(f'Total failed    : {grand_total["failed"]}')
print(f'Wall time       : {total_elapsed/60:.1f} min')

## 9. Sanity check — compare one mask to website output

Optional: visually verify that the programmatic masks match what you got from cellpose.org.
Set `SAMPLE_IMAGE` to any image you already processed manually.

In [ ]:
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

# Change these two paths to a real image and its manually-downloaded mask
SAMPLE_IMAGE = LOCAL_ROOT / 'imgs/260513_TC_Level/tiles/YOUR_IMAGE.tif'
MANUAL_MASK  = LOCAL_ROOT / 'masks_manual/YOUR_IMAGE_masks.tif'

if SAMPLE_IMAGE.exists():
    img  = tifffile.imread(str(SAMPLE_IMAGE))
    auto_mask_path = LOCAL_ROOT / f'masks/260513_TC_Level/{SAMPLE_IMAGE.stem}_masks.tif'
    auto_mask = tifffile.imread(str(auto_mask_path))

    ncols = 3 if MANUAL_MASK.exists() else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

    axes[0].imshow(img if img.ndim == 3 else img, cmap='gray')
    axes[0].set_title('Original image')
    axes[0].axis('off')

    axes[1].imshow(auto_mask, cmap='tab20b')
    axes[1].set_title(f'Auto mask ({auto_mask.max()} cells)')
    axes[1].axis('off')

    if MANUAL_MASK.exists():
        manual_mask = tifffile.imread(str(MANUAL_MASK))
        axes[2].imshow(manual_mask, cmap='tab20b')
        axes[2].set_title(f'Manual mask ({manual_mask.max()} cells)')
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('Set SAMPLE_IMAGE and MANUAL_MASK paths above to run this cell.')